In [0]:
from pyspark.sql import functions as F 

In [0]:
def load_delta(path):
    return spark.read.table(path)
    
df_base = load_delta('bronze.bid.bids')

In [0]:
df_base.distinct().select('dhfinalizacao').filter(F.col("dhfinalizacao")=="-").count()

In [0]:
df_base = df_base.withColumn(
    "dhfinalizacao",
    F.when(F.col("dhfinalizacao").isin("-"), None)
    .otherwise(F.col("dhfinalizacao"))
).withColumn(
    "dhfinalizacaostr",
    F.when(F.col("dhfinalizacaostr")=="null", None)
    .otherwise(F.col("dhfinalizacaostr"))
).withColumn(
    "boolganhou",
    F.when(F.col("boolganhou")=="null", None)
    .otherwise(F.col("boolganhou"))
).withColumn(
    "bidmotivo",
    F.when(F.col("bidmotivo")=="null", None)
    .otherwise(F.col("bidmotivo"))
).withColumn(
    "nm_concorrente",
    F.when(F.col("nm_concorrente")=="null", None)
    .otherwise(F.col("nm_concorrente"))
)

In [0]:
df_bid = df_base.select(
F.col('idbid').alias('id_bid').cast('string'),
F.to_date(F.to_timestamp("dtbid")).alias('dt_abertura_bid'),
F.to_date(F.to_timestamp("dhcadastro")).alias('dt_cadastro'),
F.col('boolbid').alias('in_bid'),
F.to_date(F.to_timestamp('dhfinalizacao')).alias('dt_finalizacao_bid'),
F.col('boolganhou').alias('in_ganhou').cast('int'),
F.col('bidmotivo').alias('ds_bid_motivo_perda'),
F.col('nm_concorrente').alias('ds_concorrencia'),
F.col('cd_id_anonimo').alias('id_contrato').cast('string'),
).withColumn("DT_PROCESSAMENTO", F.expr("current_timestamp() - INTERVAL 3 HOURS"))


In [0]:
df_bid\
    .write\
    .mode("overwrite")\
    .option("mergeSchema", True)\
    .format("delta")\
    .saveAsTable("silver.bid.entidade_bid")